# AquaTrace

Training window: **2025-05-01 to 2026-05-31**

In [ ]:
# Install / upgrade libraries.
!pip install -q --upgrade scikit-learn joblib shap

In [ ]:
import sklearn
print(sklearn.__version__)

1.9.0


In [ ]:
!pip install copernicusmarine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.1 MB/s eta 0:00:00


In [ ]:
# Districts, coordinates, date range, folder structure
import os

DISTRICTS = {
    "Palghar":    (19.70, 72.76),
    "Raigad":     (18.52, 73.20),
    "Ratnagiri":  (16.99, 73.31),
    "Sindhudurg": (16.35, 73.55),
}

START_DATE = "2025-05-01"
END_DATE   = "2026-05-31"

os.makedirs("content/data/weather", exist_ok=True)
os.makedirs("content/data/marine", exist_ok=True)
os.makedirs("content/data/sentinel", exist_ok=True)
os.makedirs("content/data/processed", exist_ok=True)

print(f"Training window: {START_DATE} to {END_DATE}")

Training window: 2025-05-01 to 2026-05-31


## Step 1 - Weather (Open-Meteo)

In [ ]:
import requests
import pandas as pd

all_data = []
for district, (lat, lon) in DISTRICTS.items():
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={START_DATE}&end_date={END_DATE}"
        "&daily=temperature_2m_mean,precipitation_sum,windspeed_10m_max"
        "&timezone=auto"
    )
    data = requests.get(url).json()
    df = pd.DataFrame(data["daily"])
    df["district"] = district
    all_data.append(df)

weather = pd.concat(all_data)
weather.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature",
    "precipitation_sum": "rainfall",
    "windspeed_10m_max": "wind_speed",
}, inplace=True)

weather.to_csv("content/data/weather/weather.csv", index=False)
print(f"Weather data saved: {weather.shape}")
weather.head()

Weather data saved: (1584, 5)


,date,temperature,rainfall,wind_speed,district
0,2025-05-01,29.3,0.0,15.7,Palghar
1,2025-05-02,29.4,0.0,15.0,Palghar
2,2025-05-03,29.1,0.0,19.2,Palghar
3,2025-05-04,29.5,0.0,18.1,Palghar
4,2025-05-05,29.8,0.0,19.7,Palghar


## Step 2 - Sea Surface Temperature (Copernicus Marine, via `open_dataset`)

In [ ]:
import copernicusmarine

# Search datasets associated with a specific product ID
catalogue = copernicusmarine.describe(contains=["SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001"])
print(catalogue)

Fetching products:  97%|█████████▋| 299/307 [00:28<00:00, 30.39it/s]WARNING:urllib3.connectionpool:Connection pool is full, discarding connection: s3.waw3-1.cloudferro.com. Connection pool size: 10

Fetching products:  99%|█████████▊| 303/307 [00:28<00:00, 22.17it/s]WARNING:urllib3.connectionpool:Connection pool is full, discarding connection: s3.waw3-1.cloudferro.com. Connection pool size: 10

Fetching products: 100%|█████████▉| 306/307 [00:29<00:00, 10.27it/s]WARNING:urllib3.connectionpool:Connection pool is full, discarding connection: s3.waw3-1.cloudferro.com. Connection pool size: 10

Fetching catalogue 1: 100%|██████████| 2/2 [00:44<00:00, 22.16s/it]


products=[CopernicusMarineProduct(title='Global Ocean OSTIA Sea Surface Temperature and Sea Ice Analysis', product_id='SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001', thumbnail_url='https://mdl-metadata.s3.waw3-1.cloudferro.com/metadata/thumbnails/SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001.jpg', description="For the Global Ocean- the OSTIA global foundation Sea Surface Temperature product provides daily gap-free maps of: Foundation Sea Surface Temperature at 0.05° x 0.05° horizontal grid resolution, using in-situ and satellite data from both infrared and microwave radiometers. \n\nThe Operational Sea Surface Temperature and Ice Analysis (OSTIA) system is run by the UK's Met Office. OSTIA uses satellite data provided by the GHRSST project together with in-situ observations to determine the sea surface temperature.\nA high resolution (1/20° - approx. 6 km) daily analysis of sea surface temperature (SST) is produced for the global ocean and some lakes.\n\n**References:**\n\n* Good, S.; Fiedler, E.

In [ ]:
import copernicusmarine
import pandas as pd

# COPERNICUS MARINE CREDENTIALS
COPERNICUS_USERNAME = "xxxx"
COPERNICUS_PASSWORD = "xxxx"

SST_DATASET_ID = "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2"

sst_frames = []
for district, (lat, lon) in DISTRICTS.items():
    ds = copernicusmarine.open_dataset(
        dataset_id=SST_DATASET_ID,
        username=COPERNICUS_USERNAME,
        password=COPERNICUS_PASSWORD,
    )
    point = ds.sel(latitude=lat, longitude=lon, method="nearest")
    point = point.sel(time=slice(START_DATE, END_DATE))
    df = point[["analysed_sst"]].to_dataframe().reset_index()
    df["district"] = district
    sst_frames.append(df[["time", "analysed_sst", "district"]])

sst = pd.concat(sst_frames).rename(columns={"time": "date"})
sst.to_csv("content/data/marine/sst.csv", index=False)
print(f"SST data saved: {sst.shape}")
sst.head()

INFO - 2026-08-26T12:23:04Z - Selected dataset version: "default"
INFO:copernicusmarine:Selected dataset version: "default"
INFO - 2026-08-26T12:23:04Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-08-26T12:24:21Z - Selected dataset version: "default"
INFO:copernicusmarine:Selected dataset version: "default"
INFO - 2026-08-26T12:24:21Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-08-26T12:25:33Z - Selected dataset version: "default"
INFO:copernicusmarine:Selected dataset version: "default"
INFO - 2026-08-26T12:25:33Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
INFO - 2026-08-26T12:26:41Z - Selected dataset version: "default"
INFO:copernicusmarine:Selected dataset version: "default"
INFO - 2026-08-26T12:26:41Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


SST data saved: (1584, 3)


,date,analysed_sst,district
0,2025-05-01,NaN,Palghar
1,2025-05-02,NaN,Palghar
2,2025-05-03,NaN,Palghar
3,2025-05-04,NaN,Palghar
4,2025-05-05,NaN,Palghar


## Step 3 - BBP & CDM

In [ ]:
import xarray as xr

bbp_cdm = xr.open_dataset("content/data/marine/bbp_cdm.nc")

print(bbp_cdm)

<xarray.Dataset> Size: 2MB
Dimensions:          (time: 13, latitude: 120, longitude: 60)
Coordinates:
  * time             (time) datetime64[ns] 104B 2025-05-01 ... 2026-05-01
  * latitude         (latitude) float32 480B 15.02 15.06 15.1 ... 19.94 19.98
  * longitude        (longitude) float32 240B 72.02 72.06 72.1 ... 74.44 74.48
Data variables:
    BBP              (time, latitude, longitude) float32 374kB ...
    BBP_uncertainty  (time, latitude, longitude) float32 374kB ...
    CDM              (time, latitude, longitude) float32 374kB ...
    CDM_uncertainty  (time, latitude, longitude) float32 374kB ...
    flags            (time, latitude, longitude) float32 374kB ...
Attributes:
    Conventions:       CF-1.11
    title:             cmems_obs-oc_glo_bgc-optics_my_l4-multi-4km_P1M
    institution:       ACRI
    source:            surface observation
    history:           Created using software developed at ACRI-ST
    contact:           servicedesk.cmems@acri-st.fr
    referenc

In [ ]:
# Convert
# Open the NetCDF file
ds_chl = xr.open_dataset("content/data/marine/bbp_cdm.nc")
# Convert to a Pandas DataFrame
df_chl = ds_chl.to_dataframe().reset_index()
# Save as CSV
df_chl.to_csv("content/data/marine/bbp_cdm.csv", index=False)
print("Successfully converted to CSV!")

Successfully converted to CSV!


In [ ]:
import pandas as pd
import numpy as np
import os

# Load raw Copernicus Marine CSV
df = pd.read_csv("content/data/marine/bbp_cdm.csv")

# Rename time column
df = df.rename(columns={"time": "date"})
df["date"] = pd.to_datetime(df["date"])

# Four Konkan district centres
DISTRICTS = {
    "Palghar": (19.6967, 72.7699),
    "Raigad": (18.5204, 73.1812),
    "Ratnagiri": (16.9902, 73.3120),
    "Sindhudurg": (16.3492, 73.5594),
}

# Assign each satellite grid point to nearest district
def nearest_district(lat, lon):
    best = None
    best_dist = float("inf")

    for district, (dlat, dlon) in DISTRICTS.items():
        dist = (lat - dlat) ** 2 + (lon - dlon) ** 2
        if dist < best_dist:
            best_dist = dist
            best = district

    return best

df["district"] = df.apply(
    lambda r: nearest_district(r["latitude"], r["longitude"]),
    axis=1
)

# Daily district averages
bbp_cdm_daily = (
    df.groupby(["date", "district"], as_index=False)[["BBP", "CDM"]]
      .mean()
)

os.makedirs("content/data/marine", exist_ok=True)
output = "content/data/marine/bbp_cdm.csv"

bbp_cdm_daily.to_csv(output, index=False)

print("Saved:", output)
print(bbp_cdm_daily.shape)
display(bbp_cdm_daily.head())

Saved: content/data/marine/bbp_cdm.csv
(52, 4)


,date,district,BBP,CDM
0,2025-05-01,Palghar,0.027354,0.267194
1,2025-05-01,Raigad,0.008834,0.106333
2,2025-05-01,Ratnagiri,0.002119,0.029691
3,2025-05-01,Sindhudurg,0.002245,0.024960
4,2025-06-01,Palghar,0.036337,0.647199


## Step 4 - Daily NDVI (Google Earth Engine, MODIS MOD13Q1) with imputation

In [ ]:
import ee
ee.Authenticate(force=True)

In [ ]:
import ee
import pandas as pd
import numpy as np

ee.Initialize(project="xxxx")   # <-- put your GCP project id here

modis_ndvi = ee.ImageCollection("MODIS/061/MOD13Q1").select("NDVI")

ndvi_frames = []
for district, (lat, lon) in DISTRICTS.items():
    point = ee.Geometry.Point([lon, lat])
    collection = modis_ndvi.filterDate(START_DATE, END_DATE).filterBounds(point)

    def extract(image):
        value = image.reduceRegion(ee.Reducer.mean(), point, 250).get("NDVI")
        return ee.Feature(None, {
            "date": image.date().format("YYYY-MM-dd"),
            "ndvi": value,
        })

    features = collection.map(extract).getInfo()["features"]
    rows = [f["properties"] for f in features]
    df = pd.DataFrame(rows)
    df["district"] = district
    ndvi_frames.append(df)

ndvi_raw = pd.concat(ndvi_frames)
ndvi_raw["date"] = pd.to_datetime(ndvi_raw["date"])
ndvi_raw["ndvi"] = ndvi_raw["ndvi"] / 10000.0  # MODIS NDVI scale factor

# Expand the sparse 16-day composites to every daily date, then impute.
full_ndvi = []
for district in DISTRICTS:
    sub = ndvi_raw[ndvi_raw["district"] == district].set_index("date").sort_index()
    daily_index = pd.date_range(START_DATE, END_DATE, freq="D")
    sub = sub.reindex(daily_index)
    sub["ndvi"] = sub["ndvi"].interpolate(method="linear").bfill().ffill()
    sub["district"] = district
    sub = sub.reset_index().rename(columns={"index": "date"})
    full_ndvi.append(sub[["date", "district", "ndvi"]])

ndvi = pd.concat(full_ndvi)
ndvi.to_csv("content/data/sentinel/daily_ndvi.csv", index=False)
print(f"Daily NDVI (imputed) saved: {ndvi.shape}")
ndvi.head()

Daily NDVI (imputed) saved: (1584, 3)


,date,district,ndvi
0,2025-05-01,Palghar,0.3549
1,2025-05-02,Palghar,0.3549
2,2025-05-03,Palghar,0.3549
3,2025-05-04,Palghar,0.3549
4,2025-05-05,Palghar,0.3549


## Step 5 - Merge weather + SST + BBP/CDM + NDVI into the master dataset

In [ ]:
import pandas as pd
import os

weather = pd.read_csv("content/data/weather/weather.csv")
sst = pd.read_csv("content/data/marine/sst.csv")
bbp_cdm = pd.read_csv("content/data/marine/bbp_cdm.csv")
ndvi = pd.read_csv("content/data/sentinel/daily_ndvi.csv")

weather["date"] = pd.to_datetime(weather["date"])
sst["date"] = pd.to_datetime(sst["date"])
bbp_cdm["date"] = pd.to_datetime(bbp_cdm["date"])
ndvi["date"] = pd.to_datetime(ndvi["date"])

# Regional daily averages (avoids row explosion from raw satellite grid points)
sst_daily = sst.groupby(["date", "district"], as_index=False)[["analysed_sst"]].mean()
bbp_cdm_daily = bbp_cdm.groupby(["date", "district"], as_index=False)[["BBP", "CDM"]].mean()

df = weather.merge(sst_daily, on=["date", "district"], how="left")
df = df.merge(bbp_cdm_daily, on=["date", "district"], how="left")
df = df.merge(ndvi, on=["date", "district"], how="left")

# Light safety-net fill for any leftover satellite gaps (cloud cover etc.);
# NDVI is already fully imputed previously
for col in ["analysed_sst", "BBP", "CDM"]:
    df[col] = df.groupby("district")[col].transform(lambda x: x.ffill().bfill())

df["month"] = df["date"].dt.month

output_path = "content/data/processed/master_konkan_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Merge successful! Final dataset shape: {df.shape}")
df.head()

Merge successful! Final dataset shape: (1584, 10)


,date,temperature,rainfall,wind_speed,district,analysed_sst,BBP,CDM,ndvi,month
0,2025-05-01,29.3,0.0,15.7,Palghar,NaN,0.027354,0.267194,0.3549,5
1,2025-05-02,29.4,0.0,15.0,Palghar,NaN,0.027354,0.267194,0.3549,5
2,2025-05-03,29.1,0.0,19.2,Palghar,NaN,0.027354,0.267194,0.3549,5
3,2025-05-04,29.5,0.0,18.1,Palghar,NaN,0.027354,0.267194,0.3549,5
4,2025-05-05,29.8,0.0,19.7,Palghar,NaN,0.027354,0.267194,0.3549,5


## Step 6 - District Baseline Category (computed from CMFRI landings, via code)

In [ ]:
import pandas as pd

CMFRI_PATH = "content/data/processed/Districtwise_and_specieswise_Maharashtra.csv"  # <-- upload this file to Colab
YEAR_SUFFIXES = ["2018-19", "2019-20", "2020-21", "2021-22", "2022-23"]
KONKAN_DISTRICTS = ["Palghar", "Raigad", "Ratnagiri", "Sindhudurg"]

cmfri = pd.read_csv(CMFRI_PATH)
totals_row = cmfri[cmfri["Sl. No."] == "Total"].iloc[0]

avg_landings = {}
for district in KONKAN_DISTRICTS:
    vals = [float(totals_row[f"{district} - {yr}"]) for yr in YEAR_SUFFIXES]
    avg_landings[district] = sum(vals) / len(vals)

baseline_df = pd.DataFrame({
    "district": list(avg_landings.keys()),
    "avg_total_landings": list(avg_landings.values()),
})

# Data-driven classification (computed via code, not hardcoded numbers):
# rank districts by their own average landings and split into 3 roughly
# equal-sized tiers using quantile-based binning.
baseline_df["baseline_category"] = pd.qcut(
    baseline_df["avg_total_landings"], q=3, labels=["Low", "Medium", "High"]
)

baseline_df.to_csv("content/data/processed/district_baseline.csv", index=False)
print("Saved district_baseline.csv")
baseline_df

Saved district_baseline.csv


,district,avg_total_landings,baseline_category
0,Palghar,49409.0,Medium
1,Raigad,44721.2,Medium
2,Ratnagiri,73846.0,High
3,Sindhudurg,17516.2,Low


## Step 7 - Label the master dataset: `baseline_category` + `fishery_risk`

In [ ]:
import pandas as pd

df = pd.read_csv("content/data/processed/master_konkan_dataset.csv", parse_dates=["date"])
baseline_df = pd.read_csv("content/data/processed/district_baseline.csv")

baseline_lookup = baseline_df.set_index("district")["baseline_category"].to_dict()
df["baseline_category"] = df["district"].map(baseline_lookup)

# Environmental stress score: +1 for each variable in its own top quartile
# that day (rainfall uses a fixed 1mm cutoff since most days are 0mm).
sst_p75 = df["analysed_sst"].quantile(0.75)
bbp_p75 = df["BBP"].quantile(0.75)
cdm_p75 = df["CDM"].quantile(0.75)
wind_p75 = df["wind_speed"].quantile(0.75)
RAIN_THRESH = 1.0

def stress_score(row):
    score = 0
    if row["analysed_sst"] >= sst_p75: score += 1
    if row["rainfall"] >= RAIN_THRESH: score += 1
    if row["wind_speed"] >= wind_p75: score += 1
    if row["BBP"] >= bbp_p75: score += 1
    if row["CDM"] >= cdm_p75: score += 1
    return score

df["env_stress_score"] = df.apply(stress_score, axis=1)

def fishery_risk(row):
    cat, score = row["baseline_category"], row["env_stress_score"]
    if cat == "High":
        return "High" if score >= 4 else "Moderate" if score >= 2 else "Low"
    elif cat == "Medium":
        return "High" if score >= 3 else "Moderate" if score >= 1 else "Low"
    else:  # Low baseline -> most vulnerable to environmental stress
        return "High" if score >= 2 else "Moderate" if score >= 1 else "Low"

df["fishery_risk"] = df.apply(fishery_risk, axis=1)

output_path = "content/data/processed/master_konkan_dataset_labeled.csv"
df.to_csv(output_path, index=False)
print(f"Labeled dataset saved: {df.shape}")
df["fishery_risk"].value_counts()

Labeled dataset saved: (1584, 13)


,count
fishery_risk,
Moderate,717
Low,604
High,263


## Step 8 - Train the RandomForest model

In [ ]:
import json

import numpy as np
import pandas as pd
import joblib
import shap

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# DISTRICT -> LAT/LON LOOKUP (for the React + Leaflet frontend map)
DISTRICT_COORDS = {
    "Palghar":    {"latitude": 19.6970, "longitude": 72.7654},
    "Raigad":     {"latitude": 18.2450, "longitude": 73.1300},
    "Ratnagiri":  {"latitude": 16.9902, "longitude": 73.3120},
    "Sindhudurg": {"latitude": 16.3667, "longitude": 73.5833},
}

# -------------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------------
DATA_PATH = "content/data/processed/master_konkan_dataset_labeled.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

# Columns present in the dataset:
# date, temperature, rainfall, wind_speed, district, analysed_sst,
# BBP, CDM, month, ndvi, baseline_category, fishery_risk

# Ensure chronological order within each district (required for lag features)
df = df.sort_values(["district", "date"]).reset_index(drop=True)

print(f"Loaded dataset with shape: {df.shape}")
print(df["fishery_risk"].value_counts(), "\n")


# -------------------------------------------------------------------
# 2. SPATIO-TEMPORAL FEATURE ENGINEERING (lag features per district)
# -------------------------------------------------------------------
# "prev_*" = value of that variable on the previous day for the SAME district.
# groupby().shift(1) keeps the lag from leaking across districts.
df["prev_sst"] = df.groupby("district")["analysed_sst"].shift(1)
df["prev_rainfall"] = df.groupby("district")["rainfall"].shift(1)
df["prev_BBP"] = df.groupby("district")["BBP"].shift(1)
df["prev_CDM"] = df.groupby("district")["CDM"].shift(1)

# The first row of each district has no "previous day" -> NaN.
# Fill using that district's own first observed value (safe, no future leakage).
lag_cols = ["prev_sst", "prev_rainfall", "prev_BBP", "prev_CDM"]
for col in lag_cols:
    df[col] = df.groupby("district")[col].transform(lambda s: s.bfill())

# Day-over-day deltas — useful signals for sudden environmental shifts
df["delta_sst"] = df["analysed_sst"] - df["prev_sst"]
df["delta_rainfall"] = df["rainfall"] - df["prev_rainfall"]
df["delta_BBP"] = df["BBP"] - df["prev_BBP"]
df["delta_CDM"] = df["CDM"] - df["prev_CDM"]

print("Spatio-temporal lag features created:", lag_cols, "\n")


# -------------------------------------------------------------------
# 3. HOTSPOT DETECTION — Z-score + Isolation Forest
# -------------------------------------------------------------------
# Environmental variables used to flag anomalous ("hotspot") observations
hotspot_features = ["analysed_sst", "rainfall", "wind_speed", "BBP", "CDM"]

# --- 3a. Z-score based hotspot flag (per-district, since baselines differ) ---
def zscore_group(series):
    mu = series.mean()
    sigma = series.std(ddof=0)
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - mu) / sigma

for feat in hotspot_features:
    df[f"z_{feat}"] = df.groupby("district")[feat].transform(zscore_group)

z_cols = [f"z_{feat}" for feat in hotspot_features]
# A row is a Z-score hotspot if ANY variable's absolute z-score exceeds 2 (≈95% CI)
df["zscore_hotspot"] = (df[z_cols].abs() > 2).any(axis=1).astype(int)

# --- 3b. Isolation Forest based hotspot flag (multivariate anomaly detection) ---
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # assume ~5% of days are environmental anomalies
    random_state=42,
)
iso_input = df[hotspot_features].fillna(df[hotspot_features].mean())
iso_forest.fit(iso_input)
# IsolationForest: -1 = anomaly (hotspot), 1 = normal -> convert to 1/0 flag
df["isoforest_hotspot"] = (iso_forest.predict(iso_input) == -1).astype(int)

# Combined hotspot flag: consider it a hotspot if EITHER method flags it
df["is_hotspot"] = ((df["zscore_hotspot"] == 1) | (df["isoforest_hotspot"] == 1)).astype(int)

print(f"Z-score hotspots flagged   : {df['zscore_hotspot'].sum()}")
print(f"Isolation Forest hotspots  : {df['isoforest_hotspot'].sum()}")
print(f"Combined hotspot days      : {df['is_hotspot'].sum()} / {len(df)}\n")


# -------------------------------------------------------------------
# 6. HARMFUL ALGAL BLOOM (HAB) DETECTION — threshold/anomaly logic
#    (We use BBP + CDM + SST + calm-wind conditions as proxies: HABs tend to develop
#    under high particulate
#    backscatter (BBP), high colored dissolved organic matter (CDM),
#    elevated sea surface temperature, and low wind/mixing that lets
#    surface water stratify.)
# -------------------------------------------------------------------
bbp_thresh = df["BBP"].quantile(0.75)
cdm_thresh = df["CDM"].quantile(0.75)
sst_thresh = df["analysed_sst"].quantile(0.75)
wind_thresh = df["wind_speed"].quantile(0.25)  # calm winds = bottom quartile

# Build a HAB susceptibility score (0-4) instead of requiring every condition
# simultaneously (an AND of four 75th-percentile cuts is almost never all true
# at once). Each favorable condition for bloom formation adds one point.
df["hab_score"] = (
    (df["BBP"] >= bbp_thresh).astype(int)
    + (df["CDM"] >= cdm_thresh).astype(int)
    + (df["analysed_sst"] >= sst_thresh).astype(int)
    + (df["wind_speed"] <= wind_thresh).astype(int)
)

# A multivariate anomaly (Isolation Forest) co-occurring with high BBP+CDM
# also bumps the score, since blooms often show up as joint anomalies.
df["hab_score"] += (
    (df["isoforest_hotspot"] == 1) & (df["BBP"] >= bbp_thresh) & (df["CDM"] >= cdm_thresh)
).astype(int)

# Convert score into a 3-tier HAB risk label
def hab_label(score):
    if score >= 2:
        return "High"
    elif score >= 1:
        return "Moderate"
    else:
        return "Low"

df["hab_risk"] = df["hab_score"].apply(hab_label)

print("Harmful Algal Bloom (HAB) risk thresholds used:")
print(f"  BBP  >= {bbp_thresh:.6f} (75th percentile)")
print(f"  CDM  >= {cdm_thresh:.6f} (75th percentile)")
print(f"  SST  >= {sst_thresh:.4f} (75th percentile)")
print(f"  Wind <= {wind_thresh:.2f} (25th percentile, calm/stratified conditions)")
print(df["hab_risk"].value_counts(), "\n")


# -------------------------------------------------------------------
# 4. TRAIN / VALIDATION / TEST SPLIT (60 / 20 / 20)
# -------------------------------------------------------------------
feature_cols = [
    "temperature", "rainfall", "wind_speed", "analysed_sst", "BBP", "CDM",
    "month", "ndvi",
    "prev_sst", "prev_rainfall", "prev_BBP", "prev_CDM",
    "delta_sst", "delta_rainfall", "delta_BBP", "delta_CDM",
    "is_hotspot",
]

# Encode categorical columns needed as model inputs
district_encoder = LabelEncoder()
df["district_enc"] = district_encoder.fit_transform(df["district"])

baseline_encoder = LabelEncoder()
df["baseline_category_enc"] = baseline_encoder.fit_transform(df["baseline_category"])

feature_cols += ["district_enc", "baseline_category_enc"]

target_encoder = LabelEncoder()
df["fishery_risk_enc"] = target_encoder.fit_transform(df["fishery_risk"])

X = df[feature_cols].copy()
y = df["fishery_risk_enc"].copy()

# Step 1: split off 60% train, 40% temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
# Step 2: split the 40% temp into 20% validation / 20% test (i.e. 50/50 of temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train shape: {X_train.shape} | Val shape: {X_val.shape} | Test shape: {X_test.shape}\n")


# -------------------------------------------------------------------
# 5. TRAIN RANDOM FOREST CLASSIFIER
# -------------------------------------------------------------------
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)

# Quick validation check (used to sanity-check before final test evaluation)
val_preds = rf_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}\n")


# -------------------------------------------------------------------
# 7. FINAL EVALUATION ON TEST SET
# -------------------------------------------------------------------
test_preds = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, test_preds)
precision = precision_score(y_test, test_preds, average="weighted", zero_division=0)
recall = recall_score(y_test, test_preds, average="weighted", zero_division=0)
f1 = f1_score(y_test, test_preds, average="weighted", zero_division=0)
conf_matrix = confusion_matrix(y_test, test_preds)

print("=" * 50)
print("TEST SET PERFORMANCE — fishery_risk classifier")
print("=" * 50)
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(f"Classes order: {list(target_encoder.classes_)}")
print(conf_matrix)
print("\nDetailed classification report:")
print(
    classification_report(
        y_test, test_preds, target_names=target_encoder.classes_, zero_division=0
    )
)

# Feature importance (useful for the SIH demo/report)
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(
    ascending=False
)
print("Top feature importances:")
print(importances.head(10), "\n")


# -------------------------------------------------------------------
# 9. SHAP EXPLANATIONS — top contributing factors PER DISTRICT
# -------------------------------------------------------------------
# SHAP tells us, for each prediction, how much each feature "pushed" the
# model's decision. Compute this for every row, then average it by
# district to answer: "In this district, what usually drives the risk?"
print("=" * 50)
print("SHAP EXPLANATIONS — top contributing factors per district")
print("=" * 50)

# TreeExplainer is the fast, exact SHAP method for tree models like RandomForest
shap_explainer = shap.TreeExplainer(rf_model)
shap_values = shap_explainer.shap_values(X)

# Depending on the SHAP/scikit-learn version, shap_values for a multi-class
if isinstance(shap_values, list):
    per_row_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
elif shap_values.ndim == 3:
    per_row_abs_shap = np.abs(shap_values).mean(axis=2)
else:
    per_row_abs_shap = np.abs(shap_values)

shap_df = pd.DataFrame(per_row_abs_shap, columns=feature_cols, index=df.index)
shap_df["district"] = df["district"].values

# "district_enc" is just a numeric ID standing in for the district
# name (e.g. Palghar=0, Raigad=1...). It is NOT a real environmental cause,
# so we exclude it here — we only show genuine contributing factors
# like SST, rainfall, wind, BBP, CDM, ndvi, etc.
EXCLUDED_FROM_SHAP_DISPLAY = {"district_enc"}
shap_display_features = [c for c in feature_cols if c not in EXCLUDED_FROM_SHAP_DISPLAY]

shap_top_factors_by_district = {}
for district in df["district"].unique():
    district_rows = shap_df[shap_df["district"] == district]
    avg_impact = district_rows[shap_display_features].mean().sort_values(ascending=False)
    top_5 = [
        {"feature": feat, "importance": round(float(val), 6)}
        for feat, val in avg_impact.head(5).items()
    ]
    shap_top_factors_by_district[district] = top_5

    factor_str = ", ".join(f"{tf['feature']}={tf['importance']:.4f}" for tf in top_5)
    print(f"  {district}: {factor_str}")

# Attach lat/lon and save as a frontend-ready JSON for the dashboard
shap_output = []
for district, top_factors in shap_top_factors_by_district.items():
    coords = DISTRICT_COORDS.get(district, {})
    shap_output.append({
        "district": district,
        "latitude": coords.get("latitude"),
        "longitude": coords.get("longitude"),
        "top_factors": top_factors,
    })

with open("shap_top_factors_by_district.json", "w") as f:
    json.dump(shap_output, f, indent=2)

print("Saved per-district SHAP factors to 'shap_top_factors_by_district.json'")
print("(Note: 'district_enc' excluded — it's a row label, not a real cause)\n")


# -------------------------------------------------------------------
# FRONTEND-READY JSON EXPORTS (for the React + Leaflet dashboard, served
# through the FastAPI backend). Each district's MOST RECENT day is treated
# as its "current status" for the map. All three outputs share the same
# district/lat/lon shape so the backend can serve them as-is or merge them.
# -------------------------------------------------------------------
# Predict fishery_risk for every row using the trained model (not just the
# test split), so the "latest day" snapshot below reflects a live model call.
df["fishery_risk_pred"] = target_encoder.inverse_transform(rf_model.predict(X))

latest_rows = df.sort_values("date").groupby("district").tail(1).set_index("district")

hotspot_output, fishery_risk_output, hab_risk_output = [], [], []

for district, coords in DISTRICT_COORDS.items():
    if district not in latest_rows.index:
        continue
    row = latest_rows.loc[district]

    hotspot_output.append({
        "district": district,
        "latitude": coords["latitude"],
        "longitude": coords["longitude"],
        "hotspot": bool(row["is_hotspot"]),  # true/false
    })
    fishery_risk_output.append({
        "district": district,
        "latitude": coords["latitude"],
        "longitude": coords["longitude"],
        "risk": row["fishery_risk_pred"],  # "Low" / "Moderate" / "High"
    })
    hab_risk_output.append({
        "district": district,
        "latitude": coords["latitude"],
        "longitude": coords["longitude"],
        "hab": row["hab_risk"],  # "Low" / "Moderate" / "High"
    })

with open("hotspot_output.json", "w") as f:
    json.dump(hotspot_output, f, indent=2)
with open("fishery_risk_output.json", "w") as f:
    json.dump(fishery_risk_output, f, indent=2)
with open("hab_risk_output.json", "w") as f:
    json.dump(hab_risk_output, f, indent=2)

print("Saved frontend-ready JSON files:")
print("  hotspot_output.json      ->", hotspot_output)
print("  fishery_risk_output.json ->", fishery_risk_output)
print("  hab_risk_output.json     ->", hab_risk_output, "\n")


# -------------------------------------------------------------------
# 8. SAVE TRAINED MODEL WITH JOBLIB
# -------------------------------------------------------------------
model_bundle = {
    "model": rf_model,
    "feature_cols": feature_cols,
    "target_encoder": target_encoder,
    "district_encoder": district_encoder,
    "baseline_encoder": baseline_encoder,
    "district_coords": DISTRICT_COORDS,  # bundled so FastAPI can reuse it directly
}

joblib.dump(model_bundle, "updated_fishery_model.pkl")
print("Trained model saved to 'updated_fishery_model.pkl'")

Loaded dataset with shape: (1584, 13)
fishery_risk
Moderate    717
Low         604
High        263
Name: count, dtype: int64 

Spatio-temporal lag features created: ['prev_sst', 'prev_rainfall', 'prev_BBP', 'prev_CDM'] 

Z-score hotspots flagged   : 293
Isolation Forest hotspots  : 80
Combined hotspot days      : 294 / 1584

Harmful Algal Bloom (HAB) risk thresholds used:
  BBP  >= 0.022308 (75th percentile)
  CDM  >= 0.267194 (75th percentile)
  SST  >= 302.6050 (75th percentile)
  Wind <= 14.30 (25th percentile, calm/stratified conditions)
hab_risk
Low         747
Moderate    446
High        391
Name: count, dtype: int64 

Train shape: (950, 19) | Val shape: (317, 19) | Test shape: (317, 19)

Validation Accuracy: 0.9937

TEST SET PERFORMANCE — fishery_risk classifier
Accuracy  : 0.9874
Precision : 0.9878
Recall    : 0.9874
F1-score  : 0.9874

Confusion Matrix (rows=actual, cols=predicted):
Classes order: ['High', 'Low', 'Moderate']
[[ 53   0   0]
 [  0 121   0]
 [  0   4 139]]

Detai

In [ ]:
import joblib

bundle = joblib.load("updated_fishery_model.pkl")
print(bundle.keys())

dict_keys(['model', 'feature_cols', 'target_encoder', 'district_encoder', 'baseline_encoder', 'district_coords'])
